# Reinforcement Learning from Verifiable Rewards (RLVR) with SageMaker

## Lab 3 — Benchmark Evaluation

In this lab, you will evaluate the RLVR fine-tuned model against the **MATH benchmark** and compare it to the base model to measure the improvement in mathematical reasoning.

### Why benchmark evaluation?

RLVR trains models on tasks with objectively verifiable answers, so it makes sense to evaluate them the same way — using a standardized benchmark with known correct answers rather than subjective LLM-as-a-Judge scoring.

### What you'll do in this notebook

1. Retrieve the fine-tuned model from the Model Registry
2. Explore available benchmarks and create a `BenchMarkEvaluator`
3. Run evaluation on both the fine-tuned and base model
4. Compare the results

---

## 1. Prerequisites

### Set up the SageMaker session

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

base_model_id = "huggingface-reasoning-qwen3-06b"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Retrieve the fine-tuned model

We look up the Model Package Group created in Lab 2 and construct the ARN for the first model version. We also set an S3 output path for the evaluation artifacts.

In [ ]:
import hashlib

MAX_MPG_NAME_LENGTH = 63
suffix = "-rlvr"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

response = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if len(response["ModelPackageSummaryList"]) > 0:
    fine_tuned_model_package_arn = response["ModelPackageSummaryList"][0]["ModelPackageArn"]
    fine_tuned_model_package_group_arn = ModelPackageGroup.get(model_package_group_name).model_package_group_arn
else:
    fine_tuned_model_package_arn = None
    fine_tuned_model_package_group_arn = None

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/evaluation"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}/evaluation"
    
if default_prefix:
    output_path_base = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/benchmark-evaluation-base"
else:
    output_path_base = f"s3://{bucket_name}/{base_model_id}/benchmark-evaluation-base"

print(f"Model Package Group: {model_package_group_name}")
print(f"Model Package Group ARN: {fine_tuned_model_package_group_arn}")
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")
print(f"Evaluation output path: {output_path}")

---

## 2. Explore available benchmarks

SageMaker provides several built-in benchmarks. Let's list them and inspect the properties of the **MATH** benchmark we'll use.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

Benchmark = get_benchmarks()
pprint(list(Benchmark))

In [ ]:
pprint(get_benchmark_properties(benchmark=Benchmark.MATH))

---

## 3. Run the evaluations, or use pre-computed results

`BenchMarkEvaluator` runs the selected benchmark against your model. It runs twice: once
for the fine-tuned model, passed as a model-package ARN, and once for the base model,
passed directly as its JumpStart model ID, so the two can be compared. Each call sets
`evaluate_base_model=False` because the base model is evaluated as its own job rather
than letting a single evaluator do both.

That takes **15-30 minutes** for the two models. You can either run it yourself or load
results produced by exactly the cells below, so the rest of the notebook is identical
either way.

Set the switch in the next cell once. Everything after it follows that setting, so
**Run All** works whichever you choose.

In [ ]:
# The one switch in this notebook.
#   False -> load pre-computed results, about a minute
#   True  -> launch the two benchmark jobs yourself, 15-30 minutes
RUN_EVALUATIONS = False

if RUN_EVALUATIONS:
    print("Will launch 2 MATH benchmark jobs (fine-tuned and base).")
    print("Expect 15-30 minutes. The notebook blocks until they finish.")
else:
    print("Will load pre-computed results from the workshop asset bucket.")
    print("No jobs are launched and nothing is billed.")
    print("Set RUN_EVALUATIONS = True above to run the evaluations instead.")

In [ ]:
import json
import os

import boto3
from sagemaker.train.common_utils.show_results_utils import (
    _display_metrics_tables,
    _extract_metrics_from_results,
)

if RUN_EVALUATIONS:
    # Evaluate the fine-tuned model
    evaluator = BenchMarkEvaluator(
        benchmark=Benchmark.MATH,
        model=fine_tuned_model_package_arn,
        model_package_group=model_package_group_name,
        base_eval_name='fine-tuned-model-rlvr',
        s3_output_path=output_path,
        evaluate_base_model=False,
        sagemaker_session=sess,
    )

    # pprint(evaluator)
    execution = evaluator.evaluate()
    # execution.wait()

    # Evaluate the base model using the JumpStart model ID directly
    base_evaluator = BenchMarkEvaluator(
        benchmark=Benchmark.MATH,
        model=base_model_id,
        base_eval_name='base-model-rlvr',
        s3_output_path=output_path_base,
        evaluate_base_model=False,
        sagemaker_session=sess,
    )

    # pprint(base_evaluator)
    execution = base_evaluator.evaluate()
    execution.wait()
else:
    os.makedirs("./eval_results", exist_ok=True)

    # Workshop assets are replicated per Region; read the replica in this session's
    # Region, since Studio's VPC endpoint refuses cross-Region S3 requests.
    ASSET_BUCKETS = {"us-east-1": "ws-assets-prod-iad-r-iad-ed304a55c2ca1aee",
                     "us-west-2": "ws-assets-prod-iad-r-pdx-f3b3f9f1a7d6a3d0"}
    asset_region = boto3.Session().region_name
    PRECOMPUTED_BUCKET = ASSET_BUCKETS[asset_region]
    PRECOMPUTED_PREFIX = "548b5be9-2da8-4c93-82f7-b0b474108ab3/lab3/eval_results"

    s3 = boto3.client("s3", region_name=asset_region)
    for name in ("base_benchmark_results.json", "tuned_benchmark_results.json"):
        s3.download_file(PRECOMPUTED_BUCKET, f"{PRECOMPUTED_PREFIX}/{name}",
                         f"./eval_results/{name}")

    with open("./eval_results/tuned_benchmark_results.json") as f:
        tuned_results = json.load(f)
    with open("./eval_results/base_benchmark_results.json") as f:
        base_results = json.load(f)
    print("downloaded:", sorted(os.listdir("./eval_results")))

---

## 4. View evaluation results

The MATH benchmark scores for the base model and the RLVR fine-tuned model, side by
side. On the live path these come from the evaluation executions; on the pre-computed
path they come from the files downloaded above. The table is the same in both cases.

In [ ]:
from rich.pretty import pprint
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

if RUN_EVALUATIONS:
    # Get all succeeded evaluations and take the first 2
    all_succeeded = [
        e for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
        if e.status.overall_status == "Succeeded"
    ]

    last_two_succeeded = all_succeeded[:2]

    # Print both
    for i, execution in enumerate(last_two_succeeded, 1):
        print(f"=== Succeeded Evaluation #{i} ===")
        pprint(execution)
        pprint(execution.show_results())
else:
    # the renderer show_results() uses, fed from disk instead of a live execution
    _display_metrics_tables(
        _extract_metrics_from_results(tuned_results),
        _extract_metrics_from_results(base_results),
        {"custom": f"s3://{PRECOMPUTED_BUCKET}/{PRECOMPUTED_PREFIX}/",
         "base": f"s3://{PRECOMPUTED_BUCKET}/{PRECOMPUTED_PREFIX}/"},
    )

---

## Summary

You've completed the RLVR lab! Here's what you accomplished:

1. **Lab 1** — Prepared the GSM8K dataset in RLVR format and registered it in the AI Registry
2. **Lab 2** — Launched a serverless RLVR training job with the `RLVRTrainer`
3. **Lab 3** — Evaluated the fine-tuned model against the MATH benchmark and compared it to the base model

The benchmark scores show how RLVR training improved the model's mathematical reasoning — without any human feedback or a separate reward model.